### Obtain papers from arxiv, and add them to example_pdfs_to_upload

In [17]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd()   # backend/
DATA_DIR = BASE_DIR / "data"
PDF_DIR = DATA_DIR / "pdfs"
DB_PATH = DATA_DIR / "database.json"

sys.path.append(str(BASE_DIR))

print("PDF_DIR:", PDF_DIR)
print("PDFs:", len(list(PDF_DIR.glob("*.pdf"))))
print("DB exists:", DB_PATH.exists())

PDF_DIR: c:\Users\marta\uerasmus\UNI\GENAI\project\GenAI-PR-2025w\backend\data\pdfs
PDFs: 20
DB exists: True


In [18]:
import json 
with open(DB_PATH, "r", encoding="utf-8") as f:
    db = json.load(f)

pdf_names = [e["pdf_name"] for e in db if "pdf_name" in e]
pdf_paths = [PDF_DIR / name for name in pdf_names if (PDF_DIR / name).exists()]

print("PDFs in DB:", len(pdf_names))
print("PDFs found on disk:", len(pdf_paths))
pdf_paths[:3]

PDFs in DB: 20
PDFs found on disk: 20


[WindowsPath('c:/Users/marta/uerasmus/UNI/GENAI/project/GenAI-PR-2025w/backend/data/pdfs/2025-11-05_Alice-Robertson_Multimodal-Machine-Learning-A-Survey-and-Taxonomy.pdf'),
 WindowsPath('c:/Users/marta/uerasmus/UNI/GENAI/project/GenAI-PR-2025w/backend/data/pdfs/2025-11-10_Chloe-Nguyen_Attention-Is-All-You-Need.pdf'),
 WindowsPath('c:/Users/marta/uerasmus/UNI/GENAI/project/GenAI-PR-2025w/backend/data/pdfs/2025-11-12_Chloe-Nguyen_Deep-Residual-Learning-for-Image-Recognition.pdf')]

In [12]:
paper_database = {
    # --- CORE ML/AI ---
    "1706.03762": {"researcher": "Chloe-Nguyen", "date": "2025-11-10"},
    "1512.03385": {"researcher": "Chloe-Nguyen", "date": "2025-11-12"},
    "2010.11929": {"researcher": "Chloe-Nguyen", "date": "2025-11-15"},
    "2005.11401": {"researcher": "Chloe-Nguyen", "date": "2025-11-20"},
    "1412.6980":  {"researcher": "Chloe-Nguyen", "date": "2025-11-22"},
    "1705.09406": {"researcher": "Alice-Robertson", "date": "2025-11-05"},

    # --- MEDICAL / CLINICAL (Pure Pathology/Modeling) ---
    "2512.04937": {"researcher": "Alice-Robertson", "date": "2025-11-26"},
    "2402.06880": {"researcher": "Bob-Martinez", "date": "2025-11-29"},
    "2511.08847": {"researcher": "Bob-Martinez", "date": "2025-12-02"},
    "2210.05880": {"researcher": "Alice-Robertson", "date": "2025-12-04"},
    "2310.03457": {"researcher": "Bob-Martinez", "date": "2025-12-07"},
    "2409.04888": {"researcher": "Alice-Robertson", "date": "2025-12-10"},
    
    # --- BRIDGE PAPERS ---
    "2103.15538": {"researcher": "Bob-Martinez", "date": "2025-12-05"},
    "2206.08826": {"researcher": "Chloe-Nguyen", "date": "2025-12-08"},
    "2411.11774": {"researcher": "Chloe-Nguyen", "date": "2025-12-12"},
    "1904.00625": {"researcher": "Bob-Martinez", "date": "2025-12-14"},
    "1809.07294": {"researcher": "Chloe-Nguyen", "date": "2025-12-16"},

    # --- TEAM / META RESEARCH ---
    "2507.12255": {"researcher": "Daniel-Fischer", "date": "2025-12-20"},
    "2411.10278": {"researcher": "Daniel-Fischer", "date": "2025-12-22"},
    "2411.05025": {"researcher": "Daniel-Fischer", "date": "2025-12-24"},
}

In [13]:
import arxiv
import os

def build_research_library(db):
    client = arxiv.Client()
    
    # 1. Define and create the target directory
    save_dir = "example_pdfs_to_upload"
    os.makedirs(save_dir, exist_ok=True)
    
    for arxiv_id, info in db.items():
        try:
            search = arxiv.Search(id_list=[arxiv_id])
            paper = next(client.results(search))
            
            # Formatting the title for a clean filename
            safe_title = "".join([c if c.isalnum() or c in " -_" else "" for c in paper.title])
            safe_title = safe_title.replace(" ", "-")

            MAX_SAFE_TITLE_LEN = len("A-Quantitative-Approach-for-Evaluating-Disease-Focus-and-Interpretability-of-Deep-Learning-Models-for-Alzheimers")
            if len(safe_title) > MAX_SAFE_TITLE_LEN:
                safe_title = safe_title[:MAX_SAFE_TITLE_LEN]
                        
            # 2. Construct the filename (without the path yet for the check)
            filename = f"{info['date']}_{info['researcher']}_{safe_title}.pdf"
            full_path = os.path.join(save_dir, filename)
            
            # 3. Check if the file already exists in that specific folder
            if not os.path.exists(full_path):
                print(f"Downloading: {paper.title}")
                # 4. Use dirpath and filename to save it correctly
                paper.download_pdf(dirpath=save_dir, filename=filename)
                print(f"Success: {full_path}")
            else:
                print(f"Exists: {full_path}")
        except Exception as e:
            print(f"Error with {arxiv_id}: {e}")

In [14]:
build_research_library(paper_database)

Downloading: Attention Is All You Need
Success: example_pdfs_to_upload\2025-11-10_Chloe-Nguyen_Attention-Is-All-You-Need.pdf
Downloading: Deep Residual Learning for Image Recognition
Success: example_pdfs_to_upload\2025-11-12_Chloe-Nguyen_Deep-Residual-Learning-for-Image-Recognition.pdf
Downloading: An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale
Success: example_pdfs_to_upload\2025-11-15_Chloe-Nguyen_An-Image-is-Worth-16x16-Words-Transformers-for-Image-Recognition-at-Scale.pdf
Downloading: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
Success: example_pdfs_to_upload\2025-11-20_Chloe-Nguyen_Retrieval-Augmented-Generation-for-Knowledge-Intensive-NLP-Tasks.pdf
Downloading: Adam: A Method for Stochastic Optimization
Success: example_pdfs_to_upload\2025-11-22_Chloe-Nguyen_Adam-A-Method-for-Stochastic-Optimization.pdf
Downloading: Multimodal Machine Learning: A Survey and Taxonomy
Success: example_pdfs_to_upload\2025-11-05_Alice-Robertson_Multim

In [15]:
print("\n==========================================")
print("Database build complete.")
print("PDF examples:", len(os.listdir("example_pdfs_to_upload")))
print("==========================================")


Database build complete.
PDF examples: 20
